# Diabetes prediction — Mahdi's model v1

**BRFSS 2015 Diabetes Health Indicators**, class-balanced with SMOTE-ENN.

This notebook pulls the resampled dataset from the shared **W&B artifact**, not from a
local file — so all three of us train on byte-identical data no matter who ran
`scripts/run_resample.py`.

Working agreement: only edit notebooks inside `notebooks/mahdi/`. Shared code belongs
in `src/`, and changes there should be discussed first.

## 1. Pull the resampled dataset from the W&B artifact

In [ ]:
# Run this first. It downloads the exact CSV produced by the SMOTE-ENN step.
# Requires: `wandb login` (once per machine). See the README.
import sys
from pathlib import Path

# Repo root, so `src` and `configs` resolve regardless of where Jupyter started.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "configs").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import wandb
import yaml

OWNER = "mahdi"

cfg = yaml.safe_load((REPO_ROOT / "configs" / "resample_smoteenn.yaml").read_text())
ARTIFACT_NAME = cfg["artifact"]["name"]          # "brfss-smoteenn-resampled"
TARGET = cfg["data"]["target_column"]            # "diabetes_binary"

run = wandb.init(
    project=cfg["wandb"]["project"],
    entity=cfg["wandb"]["entity"],
    job_type="train",
    name=f"{OWNER}-train-v1",
    tags=[OWNER, "modelling"],
)

# ":latest" always resolves to the newest version of the artifact.
# Pin an explicit version (e.g. ":v0") once you want frozen results.
artifact = run.use_artifact(f"{ARTIFACT_NAME}:latest")
artifact_dir = Path(artifact.download())

csv_path = next(artifact_dir.glob("*.csv"))
df = pd.read_csv(csv_path)

print(f"artifact : {ARTIFACT_NAME}:{artifact.version}")
print(f"file     : {csv_path.name}")
print(f"shape    : {df.shape}")
df.head()

## 2. Sanity check — confirm the class balance is post-SMOTE-ENN

In [ ]:
dist = df[TARGET].value_counts().sort_index()
print(dist.to_string())
print(f"\npositive rate: {df[TARGET].mean():.4f}")
print("(raw BRFSS is ~0.139 — if you see that here, you loaded the wrong file)")

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)
print(f"\nX: {X.shape}   y: {y.shape}")
print(f"features: {list(X.columns)}")

## 3. Train/test split

In [ ]:
from sklearn.model_selection import train_test_split

from src.utils.seed import set_seed

SEED = cfg["random_seed"]
set_seed(SEED)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f"train: {X_train.shape}   test: {X_test.shape}")

## 4. Mahdi's modelling — start here

Nothing below is implemented yet; this is the handoff point.

A couple of things worth keeping in mind as you experiment:

- **The test set here is also resampled.** SMOTE-ENN was applied to the whole dataset
  before splitting, so `X_test` contains synthetic minority rows and has had borderline
  rows removed by ENN. Metrics on it will look optimistic and will *not* reflect
  real-world performance. For a headline number, evaluate on a held-out slice of the
  original `data/raw/` CSV instead. Worth agreeing on one shared protocol across all
  three of us so the numbers are comparable.
- Log metrics with `run.log({...})` so all three runs land in the same W&B project and
  can be compared side by side.
- Call `run.finish()` when you're done with the notebook.

In [ ]:
# TODO (mahdi): model goes here.
#
# e.g.
#   from sklearn.ensemble import RandomForestClassifier
#   model = RandomForestClassifier(random_state=SEED, n_jobs=-1)
#   model.fit(X_train, y_train)
#   run.log({"val/roc_auc": ...})

In [ ]:
run.finish()